In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import MinMaxScaler

In [31]:
df = pd.read_csv(r"E:\SGP Fake Review\Dataset\reviewer_dataset.csv")

In [32]:
df.columns

Index(['reviewerID', 'avg_rating', 'total_reviews', 'rating_variance',
       'rating_entropy', 'first_review', 'last_review', 'active_days',
       'reviews/day', 'review_burst_7d', 'burst_ratio_7d', '1_star_reviews',
       '%_1Star', '2_star_reviews', '%_2Star', '3_star_reviews', '%_3Star',
       '4_star_reviews', '%_4Star', '5_star_reviews', '%_5Star',
       'unique_products_reviewed', 'total_helpful_votes',
       '%_helpful_per_review', 'avg_days_between_reviews'],
      dtype='object')

In [33]:
df['positive_ratio'] = (
    df['%_4Star'] +
    df['%_5Star']
)

df['neutral_ratio'] = (
    df['%_3Star']
)

df['negative_ratio'] = (
    df['%_1Star'] +
    df['%_2Star']
)

In [36]:
df_model = df[df['total_reviews']>=3]

In [37]:
drop_cols = ["reviewerID",
             "first_review",
             "last_review",
             "1_star_reviews",
             "2_star_reviews",
             "3_star_reviews",
             "4_star_reviews",
             "5_star_reviews",
             "%_1Star",
             "%_2Star",
             "%_3Star",
             "%_4Star",
             "%_5Star",
             "unique_products_reviewed",
             "total_reviews",
             "active_days",
             "total_helpful_votes"]
df_model = df_model.drop(columns = drop_cols)

In [38]:
df_model.columns

Index(['avg_rating', 'rating_variance', 'rating_entropy', 'reviews/day',
       'review_burst_7d', 'burst_ratio_7d', '%_helpful_per_review',
       'avg_days_between_reviews', 'positive_ratio', 'neutral_ratio',
       'negative_ratio'],
      dtype='object')

In [45]:
df_model

,avg_rating,rating_variance,rating_entropy,reviews/day,review_burst_7d,burst_ratio_7d,%_helpful_per_review,avg_days_between_reviews,positive_ratio,neutral_ratio,negative_ratio
0,3.800000,1.200000,1.370951,0.014164,3,0.600000,0.800000,88.000000,80.000000,0.000000,20.000000
1,5.000000,0.000000,0.000000,0.294118,3,0.600000,0.200000,4.000000,100.000000,0.000000,0.000000
2,5.000000,0.000000,0.000000,0.035354,2,0.285714,0.142857,32.833333,100.000000,0.000000,0.000000
3,4.800000,0.200000,0.721928,0.007874,1,0.200000,0.400000,158.500000,100.000000,0.000000,0.000000
4,4.377778,1.604040,0.979971,0.022310,3,0.066667,12.911111,45.818182,77.777778,13.333333,8.888889
...,...,...,...,...,...,...,...,...,...,...,...
9430,2.750000,2.785714,1.905639,0.002974,1,0.125000,0.500000,383.571429,37.500000,0.000000,62.500000
9431,4.000000,0.000000,0.000000,0.004525,1,0.333333,0.000000,330.500000,100.000000,0.000000,0.000000
9432,3.000000,4.000000,1.392147,0.009626,3,0.333333,3.444444,116.500000,44.444444,11.111111,44.444444
9433,4.857143,0.285714,0.371232,0.005863,4,0.285714,0.285714,183.307692,92.857143,7.142857,0.000000


In [44]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_model)

In [46]:
iso = IsolationForest(
    n_estimators=200,
    contamination=0.02,
    random_state=42
)

_ = iso.fit(X_scaled)
print("Isolation Forest trained successfully.")

Isolation Forest trained successfully.


In [47]:
pred = iso.predict(X_scaled)

In [48]:
scores = iso.decision_function(X_scaled)

In [49]:
score_scaler = MinMaxScaler((0,100))

behavior_score = score_scaler.fit_transform(
    scores.reshape(-1,1)
)

In [50]:
df_model["prediction"] = pred

df_model["decision_score"] = scores

df_model["behavior_score"] = behavior_score

In [51]:
anomalies = df_model[df_model["prediction"] == -1]

anomalies.describe().T

,count,mean,std,min,25%,50%,75%,max
avg_rating,189.0,2.776616,1.086554,1.000000,2.000000,2.666667,3.666667,5.000000
rating_variance,189.0,1.809033,1.603531,0.000000,0.500000,1.333333,3.000000,5.333333
rating_entropy,189.0,1.084239,0.571745,0.000000,0.811278,0.970951,1.521928,2.251629
reviews/day,189.0,2.083671,3.926315,0.001226,0.002631,0.009272,3.000000,30.000000
review_burst_7d,189.0,3.724868,4.300973,1.000000,1.000000,2.000000,5.000000,31.000000
burst_ratio_7d,189.0,0.595542,0.326770,0.061404,0.333333,0.500000,1.000000,1.000000
%_helpful_per_review,189.0,7.693927,27.928720,0.000000,0.142857,0.600000,2.500000,273.200000
avg_days_between_reviews,189.0,265.297053,309.843318,0.000000,0.000000,125.666667,519.500000,1223.000000
positive_ratio,189.0,33.207988,31.074333,0.000000,0.000000,33.333333,60.000000,100.000000
neutral_ratio,189.0,19.455860,24.467286,0.000000,0.000000,12.500000,33.333333,100.000000


In [52]:
normals = df_model[df_model["prediction"] == 1]

normals.describe().T

,count,mean,std,min,25%,50%,75%,max
avg_rating,9246.0,4.410475,0.545110,1.600000,4.111111,4.500000,4.857143,5.000000
rating_variance,9246.0,0.888385,1.114716,0.000000,0.142857,0.459578,1.204874,5.333333
rating_entropy,9246.0,0.888693,0.598722,0.000000,0.503258,0.918296,1.370951,2.321928
reviews/day,9246.0,0.131654,0.695652,0.001028,0.004537,0.007998,0.015480,8.000000
review_burst_7d,9246.0,2.486589,1.988442,1.000000,1.000000,2.000000,3.000000,43.000000
burst_ratio_7d,9246.0,0.340755,0.218729,0.017660,0.178571,0.285714,0.400000,1.000000
%_helpful_per_review,9246.0,1.821492,6.799868,0.000000,0.083333,0.400000,1.200000,275.375000
avg_days_between_reviews,9246.0,205.934492,201.201631,0.000000,72.630040,145.381818,269.812500,1458.000000
positive_ratio,9246.0,85.489944,17.410886,12.500000,76.190476,90.000000,100.000000,100.000000
neutral_ratio,9246.0,7.593842,11.762521,0.000000,0.000000,0.000000,14.285714,80.000000


In [66]:
df_model[(df_model['prediction'] == -1) & (df_model['reviews/day']<1)]

,avg_rating,rating_variance,rating_entropy,reviews/day,review_burst_7d,burst_ratio_7d,%_helpful_per_review,avg_days_between_reviews,positive_ratio,neutral_ratio,negative_ratio,prediction,decision_score,behavior_score
131,2.000000,1.500000,1.521928,0.002340,1,0.200000,4.400000,534.00,20.000000,0.000000,80.000000,-1,-0.017156,32.624806
143,3.400000,0.300000,0.970951,0.004739,3,0.600000,57.400000,263.50,40.000000,60.000000,0.000000,-1,-0.040737,26.249182
333,1.800000,1.200000,0.970951,0.015674,2,0.400000,7.800000,79.50,0.000000,40.000000,60.000000,-1,-0.045662,24.917560
596,2.400000,2.300000,1.370951,0.006158,3,0.600000,0.600000,202.75,20.000000,0.000000,80.000000,-1,-0.001467,36.866761
717,3.333333,2.666667,2.251629,0.206897,5,0.833333,0.166667,5.60,50.000000,16.666667,33.333333,-1,-0.001629,36.822967
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9208,1.666667,0.333333,0.918296,0.002901,1,0.333333,0.333333,516.50,0.000000,0.000000,100.000000,-1,-0.028791,29.478902
9243,1.333333,0.333333,0.918296,0.024390,1,0.333333,5.000000,61.00,0.000000,0.000000,100.000000,-1,-0.037279,27.184150
9310,2.000000,4.000000,0.811278,0.005175,2,0.500000,7.250000,257.00,25.000000,0.000000,75.000000,-1,-0.026634,30.062322
9340,2.000000,4.000000,0.811278,0.002408,1,0.250000,0.250000,553.00,25.000000,0.000000,75.000000,-1,-0.006686,35.455461


In [59]:
df_model_spare = df_model.drop(columns = [
    "avg_rating",
    "positive_ratio",
    "neutral_ratio",
    "negative_ratio"
])

In [60]:
scaler = StandardScaler()
X_scaled_spare = scaler.fit_transform(df_model_spare)

In [72]:
iso_spare = IsolationForest(
    n_estimators=200,
    contamination=0.01,
    random_state=42
)

_ = iso_spare.fit(X_scaled_spare)
print("Isolation Forest trained successfully.")

Isolation Forest trained successfully.


In [73]:
pred_spare = iso_spare.predict(X_scaled_spare)
scores_spare = iso_spare.decision_function(X_scaled_spare)
score_scaler_spare = MinMaxScaler((0,100))

behavior_score_spare = score_scaler_spare.fit_transform(
    scores_spare.reshape(-1,1)
)

In [74]:
df_model_spare["prediction"] = pred_spare

df_model_spare["decision_score"] = scores_spare

df_model_spare["behavior_score"] = behavior_score_spare

In [75]:
anomalies_spare = df_model_spare[df_model_spare["prediction"] == -1]

anomalies_spare.describe().T

,count,mean,std,min,25%,50%,75%,max
rating_variance,95.0,2.107743,1.720541,0.000000,0.571429,1.800000,3.200000,5.333333
rating_entropy,95.0,1.126636,0.613742,0.000000,0.721928,1.251629,1.584963,2.251629
reviews/day,95.0,4.036167,4.775780,0.001226,0.006407,3.000000,6.000000,30.000000
review_burst_7d,95.0,5.578947,5.352627,1.000000,2.000000,5.000000,7.000000,31.000000
burst_ratio_7d,95.0,0.768128,0.332382,0.061404,0.401299,1.000000,1.000000,1.000000
%_helpful_per_review,95.0,13.354188,38.555711,0.000000,0.000000,0.333333,1.909774,273.200000
avg_days_between_reviews,95.0,184.355400,338.147427,0.000000,0.000000,0.000000,184.227273,1223.000000
prediction,95.0,-1.000000,0.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000
decision_score,95.0,-0.036105,0.022640,-0.102459,-0.053013,-0.031369,-0.021053,-0.000129
behavior_score,95.0,16.616444,5.669403,0.000000,12.382410,17.802575,20.385718,25.625505


In [76]:
normals_spare = df_model_spare[df_model_spare["prediction"] == 1]

normals_spare.describe().T

,count,mean,std,min,25%,50%,75%,max
rating_variance,9340.0,0.894612,1.119840,0.000000,0.142857,0.470143,1.224092,5.333333
rating_entropy,9340.0,0.890230,0.598201,0.000000,0.503258,0.918296,1.370951,2.321928
reviews/day,9340.0,0.131440,0.694158,0.001028,0.004501,0.007972,0.015475,8.000000
review_burst_7d,9340.0,2.480193,1.982884,1.000000,1.000000,2.000000,3.000000,43.000000
burst_ratio_7d,9340.0,0.341564,0.218755,0.017660,0.181818,0.285714,0.400000,1.000000
%_helpful_per_review,9340.0,1.823022,6.771808,0.000000,0.086051,0.400000,1.222222,275.375000
avg_days_between_reviews,9340.0,207.355214,202.289113,0.000000,72.750000,146.000000,272.255682,1458.000000
prediction,9340.0,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000
decision_score,9340.0,0.233615,0.055583,0.000251,0.219175,0.256330,0.268214,0.296869
behavior_score,9340.0,84.159785,13.919134,25.720801,80.543640,89.848151,92.824194,100.000000


In [77]:
df_model_spare[(df_model_spare['prediction'] == -1)].head(20)

,rating_variance,rating_entropy,reviews/day,review_burst_7d,burst_ratio_7d,%_helpful_per_review,avg_days_between_reviews,prediction,decision_score,behavior_score
143,0.300000,0.970951,0.004739,3,0.600000,57.400000,263.500000,-1,-0.031369,17.802575
219,0.571429,1.298795,8.000000,8,1.000000,0.000000,0.000000,-1,-0.041347,15.303870
358,3.000000,1.370951,5.000000,5,1.000000,0.400000,0.000000,-1,-0.025382,19.301668
604,0.000000,0.000000,10.000000,10,1.000000,0.300000,0.000000,-1,-0.059690,10.710209
626,0.571429,0.591673,7.000000,7,1.000000,0.000000,0.000000,-1,-0.037551,16.254265
717,2.666667,2.251629,0.206897,5,0.833333,0.166667,5.600000,-1,-0.002005,25.155720
773,0.654545,0.684038,11.000000,11,1.000000,0.000000,0.000000,-1,-0.069371,8.286028
778,0.568182,1.483356,0.007444,2,0.166667,63.916667,146.454545,-1,-0.024866,19.431047
804,1.839286,2.155639,0.005145,1,0.125000,98.125000,222.000000,-1,-0.059851,10.670002
869,3.000000,1.370951,5.000000,5,1.000000,0.600000,0.000000,-1,-0.026539,19.012101


In [78]:
df_model["%_helpful_per_review"].describe()

count    9435.000000
mean        1.939128
std         7.844294
min         0.000000
25%         0.083333
50%         0.400000
75%         1.230769
max       275.375000
Name: %_helpful_per_review, dtype: float64

In [79]:
df_model["%_helpful_per_review"].quantile([0.90, 0.95, 0.99, 0.995, 0.999])

0.900     3.600000
0.950     7.205217
0.990    27.920000
0.995    46.474286
0.999    95.714750
Name: %_helpful_per_review, dtype: float64